# 01. 데이터 점검 (Data Audit)

이 노트북은 `pyrolysis.config` 와 `pyrolysis.io` 의 공통 함수를 호출한다.  
학습 알고리즘이나 전처리 코드를 이 노트북에 복사하지 않는다.  
CLI `python -m pyrolysis.cli audit` 과 **같은 공통 함수**를 호출하므로 결과가 같다.

**커밋 전: Kernel → Restart & Clear Output 후 저장.** 실제 연구 셀 출력을 저장하지 않는다.

노트북 실행만으로 GitHub push·배포는 발생하지 않는다.

In [ ]:
from pathlib import Path

from pyrolysis.config import load_problem
from pyrolysis.io import DatasetReadError, audit_dataset, read_training_file

# 설정 경로. 합성 설정 점검은 configs/demo.yaml,
# 실제 설정은 configs/local/problem.yaml (작업 26에서 확정)
problem_path = Path('configs/demo.yaml')

# 실제 학습 파일 경로·시트·식별자 열은 담당자와 확인 후 지정한다.
# dataset_path = Path('data/local/<파일>.xlsx')
# id_columns = ['source_id', 'sample_id', 'measurement_id']
dataset_path = None  # 실제 자료 수령 후 지정 (작업 25)
id_columns = []

In [ ]:
spec = load_problem(problem_path)

if dataset_path is None:
    print('실제 학습 파일 경로가 지정되지 않았습니다. 담당자와 확인 후 dataset_path 를 설정하세요.')
else:
    try:
        data = read_training_file(dataset_path, spec, sheet_name=0, id_columns=id_columns)
        report = audit_dataset(data, spec)
        print('전체 행:', report.n_total_rows)
        print('반복 측정 후보:', report.n_replicated_rows, '행')
        for tr in report.target_results:
            print(f'  {tr.target_id} [{tr.task}]: {tr.n_available}/{tr.n_total} 사용 가능')
    except DatasetReadError as exc:
        print('파일 읽기 실패:', exc)

같은 점검을 CLI 로도 실행할 수 있다:

```powershell
python -m pyrolysis.cli audit --problem configs/demo.yaml --dataset data/local/<파일>.xlsx --id-columns source_id,sample_id,measurement_id
```